# Module 4: RLHF & DPO Training

**Alwajeez Tech** | Agentic AI System Training Program

---

## Overview

In this lab, we will learn how to **align** a language model with human preferences using
**Direct Preference Optimization (DPO)**. This is a critical step in building production-quality
AI systems: after pre-training gives the model general knowledge, and fine-tuning gives it
domain expertise, **alignment** ensures the model produces responses that humans actually prefer.

### What is RLHF (Reinforcement Learning from Human Feedback)?

RLHF is a paradigm for training language models to behave in ways that align with human
preferences. The traditional RLHF pipeline has three stages:

1. **Supervised Fine-Tuning (SFT)**: Train the model on high-quality demonstrations
2. **Reward Model Training**: Train a separate model to predict human preferences
3. **RL Optimization (PPO)**: Use the reward model to optimize the language model via
   Proximal Policy Optimization

This pipeline is powerful but complex. It requires training and maintaining multiple models,
dealing with RL instabilities, and significant compute resources.

### What is DPO (Direct Preference Optimization)?

DPO, introduced by Rafailov et al. (2023), provides an elegant alternative to the full RLHF
pipeline. The key insight is:

> **You don't need a separate reward model.** The language model itself implicitly defines a
> reward function through its log-probabilities.

DPO reparameterizes the reward function in terms of the policy (language model), eliminating
the need for a separate reward model and RL training loop entirely.

### The DPO Loss Function

The DPO loss is derived from the Bradley-Terry preference model. Given a prompt $x$,
a chosen response $y_w$ (preferred), and a rejected response $y_l$ (not preferred), the loss is:

$$\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w | x)}{\pi_{\text{ref}}(y_w | x)} - \beta \log \frac{\pi_\theta(y_l | x)}{\pi_{\text{ref}}(y_l | x)} \right) \right]$$

Where:
- $\pi_\theta$ is the **policy model** (the model being trained)
- $\pi_{\text{ref}}$ is the **reference model** (frozen copy of the initial model)
- $\beta$ is a temperature parameter controlling how much the policy can diverge from the reference
- $\sigma$ is the sigmoid function

**Intuition:** The loss encourages the model to increase the probability of chosen responses
and decrease the probability of rejected responses, relative to the reference model. The
reference model acts as an anchor to prevent the policy from deviating too far.

### The Preference Learning Paradigm

Instead of absolute ratings, DPO works with **relative preferences**:

| Component | Description |
|-----------|-------------|
| **Prompt** | The input question or instruction |
| **Chosen** | The response humans prefer (accurate, specific, well-structured) |
| **Rejected** | The response humans do NOT prefer (vague, incomplete, incorrect) |

This is often more natural for annotators -- it's easier to say "A is better than B" than
to assign an absolute score.

### Why DPO over PPO?

| Aspect | DPO | PPO |
|--------|-----|-----|
| Reward Model | Not needed | Required (separate model) |
| Training Loop | Standard supervised | RL loop (generate -> score -> update) |
| Stability | Very stable | Can be unstable (reward hacking) |
| Compute | ~2x SFT | ~4-10x SFT |
| Data | Offline preferences | Can use online generation |

### What We'll Build

In this lab, we will:
1. Create a preference dataset for our Alwajeez AI system domain
2. Load and format data for DPO training
3. Train a model using DPO with QLoRA on a T4 GPU
4. Compare DPO-trained vs base model responses
5. Evaluate alignment quality with quantitative metrics
6. Understand PPO as an alternative approach
7. Build rule-based reward functions for evaluation

**Runtime:** Google Colab with T4 GPU (~15GB VRAM)

**Prerequisites:** Completion of Module 3 (QLoRA Fine-Tuning)

In [ ]:
# ============================================================
# Section 1: Google Colab Setup
# ============================================================

# Check GPU availability
!nvidia-smi

In [ ]:
# Install required libraries
# - torch: PyTorch deep learning framework
# - transformers: Hugging Face model loading and inference
# - peft: Parameter-Efficient Fine-Tuning (LoRA/QLoRA)
# - datasets: Hugging Face dataset loading utilities
# - accelerate: Distributed training and mixed precision
# - bitsandbytes: 4-bit quantization support
# - trl: Transformer Reinforcement Learning (DPO, PPO trainers)
!pip install -q torch transformers peft datasets accelerate bitsandbytes trl

In [ ]:
# Verify GPU and library versions
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

import transformers, peft, trl
print(f"\nLibrary versions:")
print(f"  PyTorch:      {torch.__version__}")
print(f"  Transformers: {transformers.__version__}")
print(f"  PEFT:         {peft.__version__}")
print(f"  TRL:          {trl.__version__}")

---

## Section 2: Create Preference Dataset

### Understanding Preference Data

The foundation of DPO training is a **preference dataset**. Each example contains:

- **prompt**: The question or instruction given to the model
- **chosen**: The response that a human annotator considers **better** (more accurate, more
  specific, better structured, more helpful)
- **rejected**: The response that is **worse** (vague, incomplete, less accurate, less helpful)

### What Makes a "Chosen" Response Better?

For our Alwajeez AI system, a good response should be:

1. **Accurate**: Contains correct facts (e.g., VAT is 15%, not 10%)
2. **Specific**: Includes concrete details (port numbers, file names, specific steps)
3. **Well-structured**: Uses numbered lists, clear organization
4. **Complete**: Covers all relevant aspects of the question
5. **Domain-aware**: References our specific system components (FastAPI, Ollama, n8n)

A "rejected" response typically:
- Uses vague language ("stuff", "things", "basically")
- Lacks specific details
- Is too short to be useful
- May contain subtle inaccuracies

### Data Quality is Critical

The quality of your preference data directly determines the quality of alignment.
Even 10 high-quality preference pairs can meaningfully shift model behavior, especially
with small models and QLoRA training.

In production systems, you would collect thousands of preference pairs from:
- Human annotators comparing model outputs
- User feedback (thumbs up/down on responses)
- Expert review of critical domain responses

In [ ]:
# ============================================================
# Section 2: Create Preference Dataset
# ============================================================

# Define 10 preference triples for our Alwajeez AI domain
# Each triple has a prompt, a high-quality chosen response, and a low-quality rejected response

preference_data = [
    {
        "prompt": "What is ZATCA?",
        "chosen": (
            "ZATCA stands for Zakat, Tax and Customs Authority, the Saudi Arabian government "
            "body responsible for collecting zakat (Islamic tax), enforcing tax regulations "
            "including VAT at 15%, and managing customs operations. ZATCA mandates electronic "
            "invoicing (e-invoicing) through two phases: Phase 1 (Generation, Dec 2021) "
            "requires generating invoices in XML format with QR codes, and Phase 2 "
            "(Integration, Jan 2023+) requires real-time integration with ZATCA's Fatoora "
            "platform for invoice clearance."
        ),
        "rejected": (
            "ZATCA is a government organization in Saudi Arabia that deals with taxes and "
            "stuff. They handle invoicing and customs. Businesses need to follow their rules."
        ),
    },
    {
        "prompt": "How does the ReAct agent work in our system?",
        "chosen": (
            "The ReAct agent in the Agentic AI System implements a Thought-Action-Observation "
            "loop defined in the ReActAgent class in agentic_server.py. The process works as "
            "follows: 1) The agent receives a task via POST /v1/agent/run. 2) It constructs a "
            "system prompt listing available tools. 3) In each iteration, it calls Ollama for "
            "inference, producing a THOUGHT (reasoning). 4) It parses any ACTION: "
            "tool_name({params}) from the response. 5) The ToolExecutor runs the tool and "
            "returns the result as an OBSERVATION. 6) The loop continues until FINAL ANSWER "
            "is produced or MAX_ITERATIONS (default 10) is reached. The response includes all "
            "steps, tool calls, and the final answer."
        ),
        "rejected": (
            "The ReAct agent is an AI that can think and act. It uses tools and tries to "
            "answer questions. It goes through several steps to find the answer."
        ),
    },
    {
        "prompt": "What is the VAT rate in Saudi Arabia?",
        "chosen": (
            "The standard VAT rate in Saudi Arabia is 15%, increased from 5% to 15% in July "
            "2020. There are three VAT categories: 1) Standard rated (15%): Most goods and "
            "services. 2) Zero-rated (0%): Exports of goods and services, international "
            "transportation, supplies of certain precious metals for investment. 3) Exempt: "
            "Certain financial services, residential real estate rentals, local passenger "
            "transport, and life insurance premiums. Businesses exceeding SAR 375,000 in "
            "annual revenue must register for VAT."
        ),
        "rejected": (
            "The VAT in Saudi Arabia is 15%. It used to be lower but they changed it."
        ),
    },
    {
        "prompt": "How do I deploy the system?",
        "chosen": (
            "Deploy the Agentic AI System using Docker Compose with these steps: 1) Ensure "
            "Docker, Docker Compose, and NVIDIA Container Toolkit (for GPU) are installed. "
            "2) Clone the repository and navigate to the project root. 3) Run "
            "'docker-compose up -d' to start all four services: Ollama (port 11434), FastAPI "
            "API (port 8000), n8n (port 5678), and Redis (port 6379). 4) The model-loader "
            "container automatically pulls llama3.2:3b on first startup. 5) Verify deployment "
            "with 'curl http://localhost:8000/health' - check that ollama status is 'healthy'. "
            "6) Access n8n at http://localhost:5678 (default credentials: admin/agentic123). "
            "Minimum requirements: 8GB RAM, 20GB disk. Recommended: 16GB RAM + NVIDIA GPU "
            "with 4GB+ VRAM."
        ),
        "rejected": (
            "Just use Docker to deploy everything. Run docker-compose up and it should work. "
            "Make sure you have Docker installed first."
        ),
    },
    {
        "prompt": "What tools does the AI agent have access to?",
        "chosen": (
            "The Agentic AI System provides seven built-in tools defined in agentic_server.py: "
            "1) calculator - evaluates mathematical expressions using safe character filtering "
            "(digits, operators, parentheses). 2) web_search - searches the internet for "
            "current information (requires search API integration). 3) get_current_time - "
            "returns the current date/time in a specified timezone (default: Asia/Riyadh). "
            "4) file_operations - reads, writes, and lists files restricted to ./data, "
            "./output, ./temp directories with a 5000-char read limit. 5) http_request - "
            "makes GET/POST/PUT/DELETE requests to external APIs. 6) zatca_compliance - "
            "validates, generates, and checks status of ZATCA e-invoices. 7) send_webhook - "
            "sends JSON payloads to n8n webhook endpoints at "
            "http://localhost:5678/webhook/{name}."
        ),
        "rejected": (
            "The agent has tools like a calculator, web search, file reading, and some other "
            "ones. It can do math and search the web and stuff."
        ),
    },
    {
        "prompt": "How do I add a custom tool to the system?",
        "chosen": (
            "To add a custom tool to the Agentic AI System, follow these steps: 1) In "
            "agentic_server.py, add a ToolDefinition to the BUILTIN_TOOLS dictionary with "
            "name, description, and a JSON Schema parameters object specifying type, "
            "properties, and required fields. 2) Add a handler method to the ToolExecutor "
            "class (e.g., 'async def _my_tool(self, param1: str) -> str'). 3) Add an "
            "'elif tool_name == \"my_tool\"' branch in the ToolExecutor.execute() method that "
            "calls your handler with the appropriate arguments. 4) Restart the FastAPI server. "
            "The agent automatically includes all BUILTIN_TOOLS in its system prompt, so it "
            "will be aware of the new tool. Make sure the tool description clearly explains "
            "when the agent should use it and what parameters it accepts."
        ),
        "rejected": (
            "You can add tools by modifying the code. Just add a new function and register "
            "it in the tools dictionary."
        ),
    },
    {
        "prompt": "What is the difference between the /v1/chat/completions and /v1/agent/run endpoints?",
        "chosen": (
            "These two endpoints serve different purposes: /v1/chat/completions (POST) is a "
            "direct LLM inference endpoint compatible with the OpenAI API format. It accepts "
            "a list of messages (system, user, assistant roles), forwards them to Ollama, and "
            "returns the model's response. No tool calling or reasoning loop is involved. "
            "Response time is fast (single LLM call). Use it for simple Q&A, translation, "
            "summarization. /v1/agent/run (POST) implements the full ReAct agent loop. It "
            "accepts a task description, optional context, and optional tool filter. The agent "
            "reasons step by step (THOUGHT), calls tools (ACTION), observes results "
            "(OBSERVATION), and iterates up to MAX_ITERATIONS times until producing a FINAL "
            "ANSWER. Use it for complex tasks requiring tool use, calculations, data "
            "retrieval, or multi-step reasoning. The response includes all steps, tool calls, "
            "and iteration count."
        ),
        "rejected": (
            "The chat endpoint is for chatting and the agent endpoint is for running tasks. "
            "The agent one is more powerful because it can use tools."
        ),
    },
    {
        "prompt": "Explain how n8n workflows connect to the AI agent.",
        "chosen": (
            "n8n integrates with the Agentic AI System bidirectionally: 1) n8n-to-Agent: n8n "
            "workflows trigger the agent via HTTP Request nodes. They can POST to "
            "/v1/agent/run with a task payload, or POST to /webhook/n8n with an event payload "
            "containing 'event': 'agent_task' and task details in 'data'. The webhook endpoint "
            "automatically creates an AgentTask and runs the ReAct loop. 2) Agent-to-n8n: The "
            "agent uses the send_webhook tool to push data to n8n. It sends POST requests to "
            "http://n8n:5678/webhook/{webhook_name} (Docker) or "
            "http://localhost:5678/webhook/{webhook_name} (local) with a JSON payload. n8n "
            "receives this via Webhook trigger nodes and can then execute downstream workflow "
            "steps like sending emails, updating databases, or calling external APIs. n8n is "
            "accessible at port 5678 with default credentials admin/agentic123."
        ),
        "rejected": (
            "n8n connects through webhooks. The agent can send data to n8n and n8n can call "
            "the agent. They talk over HTTP."
        ),
    },
    {
        "prompt": "What is fine-tuning and when should I use it?",
        "chosen": (
            "Fine-tuning is the process of further training a pre-trained LLM on "
            "domain-specific data to improve its performance on specialized tasks. For the "
            "Agentic AI System, you should consider fine-tuning when: 1) The base model "
            "(llama3.2:3b) doesn't have knowledge of your specific domain (e.g., ZATCA "
            "regulations, company-specific processes). 2) You need consistent output "
            "formatting (e.g., always responding in a specific structure). 3) You want to "
            "improve tool-calling accuracy for your specific tools. The recommended approach "
            "is QLoRA (Quantized LoRA), which loads the base model in 4-bit quantization and "
            "trains small adapter matrices on your data. This requires only a T4 GPU (16GB "
            "VRAM) and 15-20 high-quality instruction/output examples. After training, merge "
            "the adapter with the base model, convert to GGUF format, and import into Ollama "
            "using a Modelfile for local inference."
        ),
        "rejected": (
            "Fine-tuning means training a model on your own data to make it work better. You "
            "should do it when the base model doesn't know about your stuff."
        ),
    },
    {
        "prompt": "What are the system health check endpoints?",
        "chosen": (
            "The Agentic AI System exposes a comprehensive health check at GET /health that "
            "returns: 1) 'status': overall system status ('healthy' or 'degraded'). "
            "2) 'ollama': connection status to Ollama at http://ollama:11434 (Docker) or "
            "http://localhost:11434 (local), with model availability check. 3) 'timestamp': "
            "current server time. 4) 'version': API version string. The health endpoint uses "
            "httpx.AsyncClient to asynchronously check Ollama connectivity with a 5-second "
            "timeout. If Ollama is unreachable, the status changes to 'degraded' but the API "
            "still responds, allowing monitoring systems to detect partial failures. For "
            "Docker deployments, this endpoint is used as the Docker health check command "
            "(HEALTHCHECK in Dockerfile) with 30-second intervals."
        ),
        "rejected": (
            "There's a health endpoint that checks if everything is working. It tells you "
            "if the system is healthy or not."
        ),
    },
]

print(f"Created {len(preference_data)} preference triples")
print(f"\nPrompts:")
for i, item in enumerate(preference_data):
    print(f"  {i+1}. {item['prompt']}")

In [ ]:
# Save preference data as JSONL (JSON Lines) format
# Each line is a separate JSON object - this is the standard format for preference datasets

import json
import os

os.makedirs("data", exist_ok=True)
output_path = "data/preference_data.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for item in preference_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved {len(preference_data)} preference triples to {output_path}")
print(f"File size: {os.path.getsize(output_path):,} bytes")

In [ ]:
# Print dataset statistics and a sample triple

import json

# Compute statistics
chosen_lengths = [len(item["chosen"].split()) for item in preference_data]
rejected_lengths = [len(item["rejected"].split()) for item in preference_data]
prompt_lengths = [len(item["prompt"].split()) for item in preference_data]

print("=" * 60)
print("PREFERENCE DATASET STATISTICS")
print("=" * 60)
print(f"Total examples:          {len(preference_data)}")
print(f"")
print(f"Prompt length (words):   avg={sum(prompt_lengths)/len(prompt_lengths):.0f}, "
      f"min={min(prompt_lengths)}, max={max(prompt_lengths)}")
print(f"Chosen length (words):   avg={sum(chosen_lengths)/len(chosen_lengths):.0f}, "
      f"min={min(chosen_lengths)}, max={max(chosen_lengths)}")
print(f"Rejected length (words): avg={sum(rejected_lengths)/len(rejected_lengths):.0f}, "
      f"min={min(rejected_lengths)}, max={max(rejected_lengths)}")
print(f"")
print(f"Avg chosen/rejected length ratio: "
      f"{sum(chosen_lengths)/sum(rejected_lengths):.1f}x")

# Show a sample triple
print(f"\n{'=' * 60}")
print("SAMPLE TRIPLE (Example 1)")
print(f"{'=' * 60}")
sample = preference_data[0]
print(f"\nPROMPT: {sample['prompt']}")
print(f"\nCHOSEN (preferred - {len(sample['chosen'].split())} words):")
print(f"  {sample['chosen'][:200]}...")
print(f"\nREJECTED (not preferred - {len(sample['rejected'].split())} words):")
print(f"  {sample['rejected']}")

---

## Section 3: Load & Format for DPO

### DPO Data Format Requirements

The TRL `DPOTrainer` expects data in a specific format:

- **prompt**: The input formatted as a chat template string (including system message and
  user turn, but stopping before the assistant's response)
- **chosen**: The preferred completion (just the assistant's response text)
- **rejected**: The non-preferred completion (just the assistant's response text)

The chat template formatting is crucial because:
1. It matches how the model was pre-trained (with special tokens)
2. The model learns to associate the formatted prompt with the quality of its completion
3. Different models use different chat templates (ChatML, Llama, Zephyr, etc.)

For TinyLlama (which uses the Zephyr chat template), the format is:
```
<|system|>
You are Alwajeez AI, a helpful assistant.</s>
<|user|>
{question}</s>
<|assistant|>
```

The model then generates its response after `<|assistant|>`.

In [ ]:
# ============================================================
# Section 3: Load & Format for DPO
# ============================================================

from datasets import load_dataset, Dataset

# Load the JSONL file as a Hugging Face Dataset
dataset = load_dataset("json", data_files="data/preference_data.jsonl", split="train")

print(f"Loaded dataset: {dataset}")
print(f"Columns: {dataset.column_names}")
print(f"Number of examples: {len(dataset)}")


def format_for_dpo(example):
    """
    Format a preference example for DPO training.
    
    The prompt is formatted with the TinyLlama/Zephyr chat template,
    while chosen and rejected remain as plain text completions.
    The DPOTrainer will handle tokenization and padding.
    """
    formatted_prompt = (
        f"<|system|>\n"
        f"You are Alwajeez AI, a helpful assistant for Alwajeez Tech's Agentic AI System. "
        f"You provide accurate, specific, and well-structured answers about ZATCA compliance, "
        f"VAT regulations, and the system's technical architecture.</s>\n"
        f"<|user|>\n"
        f"{example['prompt']}</s>\n"
        f"<|assistant|>\n"
    )
    return {
        "prompt": formatted_prompt,
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }


# Apply formatting to all examples
dpo_dataset = dataset.map(format_for_dpo)

print(f"\nFormatted dataset: {dpo_dataset}")

In [ ]:
# Print a formatted example to verify the structure

example = dpo_dataset[0]

print("=" * 60)
print("FORMATTED DPO EXAMPLE")
print("=" * 60)

print(f"\n--- PROMPT (formatted with chat template) ---")
print(example["prompt"])

print(f"--- CHOSEN COMPLETION ---")
print(example["chosen"][:200] + "...")

print(f"\n--- REJECTED COMPLETION ---")
print(example["rejected"])

In [ ]:
# Split into train and eval sets (80/20)
# We use 8 examples for training and 2 for evaluation

split_dataset = dpo_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Train set: {len(train_dataset)} examples")
print(f"Eval set:  {len(eval_dataset)} examples")
print(f"\nTrain prompts:")
for i, ex in enumerate(train_dataset):
    # Extract the user question from the formatted prompt
    prompt_text = ex["prompt"].split("<|user|>\n")[1].split("</s>")[0]
    print(f"  {i+1}. {prompt_text}")
print(f"\nEval prompts:")
for i, ex in enumerate(eval_dataset):
    prompt_text = ex["prompt"].split("<|user|>\n")[1].split("</s>")[0]
    print(f"  {i+1}. {prompt_text}")

---

## Section 4: Load Model + Reference Model

### Why DPO Needs Two Models

DPO training requires two models:

1. **Policy Model** ($\pi_\theta$): This is the model we are training. Its parameters get
   updated during training to prefer chosen responses over rejected ones.

2. **Reference Model** ($\pi_{\text{ref}}$): This is a frozen copy of the initial model. It
   serves as a baseline to prevent the policy from changing too drastically.

The reference model is critical because without it, the policy could "cheat" by simply
assigning very high probability to chosen responses and very low probability to rejected
responses -- potentially destroying the model's general capabilities. The KL divergence
penalty (controlled by $\beta$) keeps the policy close to the reference.

### Memory-Efficient Setup with QLoRA

On a T4 GPU (15GB VRAM), we need to be memory-efficient. Our approach:
- Load the base model in **4-bit quantization** (QLoRA) to reduce memory
- Use **LoRA adapters** so only a small number of parameters are trainable
- The `DPOTrainer` from TRL handles the reference model automatically -- when using
  PEFT/LoRA, it uses the base (non-adapter) weights as the reference model, so we don't
  need to load a separate copy

In [ ]:
# ============================================================
# Section 4: Load Model + Reference Model
# ============================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

# We use TinyLlama-1.1B-Chat -- small enough for T4, good for learning
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization configuration (same as Module 3)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                  # Load weights in 4-bit
    bnb_4bit_quant_type="nf4",          # NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16,  # Compute in fp16
    bnb_4bit_use_double_quant=True,     # Quantize the quantization constants too
)

print("Loading policy model (4-bit quantized)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for k-bit training
# This handles gradient checkpointing and other QLoRA necessities
model = prepare_model_for_kbit_training(model)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set padding token
tokenizer.padding_side = "left"  # Left-padding is important for DPO (generation-focused)

print(f"Model loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")
print(f"Pad token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"Padding side: {tokenizer.padding_side}")

In [ ]:
# LoRA configuration for DPO
# We target all the attention and MLP projection layers for maximum expressiveness

peft_config = LoraConfig(
    r=16,                    # Rank of the low-rank matrices
    lora_alpha=32,           # Scaling factor (effective lr = alpha/r * lr)
    target_modules=[         # Which layers to add LoRA adapters to
        "q_proj",            # Query projection (attention)
        "k_proj",            # Key projection (attention)
        "v_proj",            # Value projection (attention)
        "o_proj",            # Output projection (attention)
        "gate_proj",         # Gate projection (MLP / SwiGLU)
        "up_proj",           # Up projection (MLP)
        "down_proj",         # Down projection (MLP)
    ],
    lora_dropout=0.05,       # Dropout on LoRA layers for regularization
    bias="none",             # Don't train bias terms
    task_type="CAUSAL_LM",   # Causal language modeling task
)

# Calculate trainable parameters
# With r=16 and 7 target modules across all layers, this is roughly:
# 22 layers * 7 modules * 2 * (hidden_dim * r) parameters
print(f"LoRA config:")
print(f"  Rank (r):        {peft_config.r}")
print(f"  Alpha:           {peft_config.lora_alpha}")
print(f"  Scaling factor:  {peft_config.lora_alpha / peft_config.r}")
print(f"  Target modules:  {peft_config.target_modules}")
print(f"  Dropout:         {peft_config.lora_dropout}")

In [ ]:
# Test the base model (before DPO training) on a sample prompt
# This gives us a baseline to compare against after training

def generate_response(model, tokenizer, prompt, max_new_tokens=200):
    """Generate a response from the model given a formatted prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the generated tokens (not the input prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# Format a test prompt
test_prompt = (
    "<|system|>\n"
    "You are Alwajeez AI, a helpful assistant for Alwajeez Tech's Agentic AI System. "
    "You provide accurate, specific, and well-structured answers about ZATCA compliance, "
    "VAT regulations, and the system's technical architecture.</s>\n"
    "<|user|>\n"
    "What is ZATCA and what are its e-invoicing phases?</s>\n"
    "<|assistant|>\n"
)

print("Base model response (before DPO):")
print("-" * 40)
base_response = generate_response(model, tokenizer, test_prompt)
print(base_response)

---

## Section 5: DPO Training

### Understanding the DPO Training Process

During DPO training, for each batch of (prompt, chosen, rejected) triples:

1. **Forward pass on chosen**: Compute $\log \pi_\theta(y_w | x)$ -- the log-probability of
   the chosen response under the current policy
2. **Forward pass on rejected**: Compute $\log \pi_\theta(y_l | x)$ -- the log-probability of
   the rejected response under the current policy
3. **Reference model scores**: Compute $\log \pi_{\text{ref}}(y_w | x)$ and
   $\log \pi_{\text{ref}}(y_l | x)$ using the frozen reference model
4. **Compute implicit rewards**:
   - $\hat{r}(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$
5. **Compute loss**: $\mathcal{L} = -\log \sigma(\hat{r}(x, y_w) - \hat{r}(x, y_l))$
6. **Backpropagate**: Update only the LoRA adapter weights

### Key Hyperparameters

- **beta** ($\beta = 0.1$): Controls the strength of the KL penalty. Higher beta means the
  model stays closer to the reference. Typical range: 0.1 - 0.5.
  - $\beta = 0.1$: Allow more divergence from reference (more alignment, risk of overfitting)
  - $\beta = 0.5$: Stay closer to reference (less alignment, more stable)
- **learning_rate** ($5 \times 10^{-5}$): Standard for DPO with LoRA
- **max_length** (512): Maximum total sequence length (prompt + completion)
- **max_prompt_length** (256): Maximum tokens for the prompt portion

In [ ]:
# ============================================================
# Section 5: DPO Training
# ============================================================

from trl import DPOTrainer, DPOConfig

# DPO training configuration
dpo_config = DPOConfig(
    output_dir="./dpo_results",            # Where to save checkpoints
    num_train_epochs=3,                     # Number of passes through the training data
    per_device_train_batch_size=2,          # Batch size per GPU (small for T4 memory)
    gradient_accumulation_steps=4,          # Effective batch size = 2 * 4 = 8
    learning_rate=5e-5,                     # Learning rate for LoRA adapter weights
    lr_scheduler_type="cosine",             # Cosine annealing for smooth decay
    warmup_ratio=0.1,                       # 10% of steps for learning rate warmup
    fp16=True,                              # Mixed precision training (saves memory)
    logging_steps=2,                        # Log metrics every 2 steps
    save_strategy="epoch",                  # Save checkpoint after each epoch
    beta=0.1,                               # DPO temperature / KL penalty strength
    max_length=512,                         # Max total sequence length
    max_prompt_length=256,                  # Max prompt length (rest is for completion)
    remove_unused_columns=False,            # Keep all dataset columns
    report_to="none",                       # Disable wandb/tensorboard for simplicity
)

print("DPO Training Configuration:")
print(f"  Epochs:              {dpo_config.num_train_epochs}")
print(f"  Batch size:          {dpo_config.per_device_train_batch_size}")
print(f"  Gradient accum:      {dpo_config.gradient_accumulation_steps}")
print(f"  Effective batch:     {dpo_config.per_device_train_batch_size * dpo_config.gradient_accumulation_steps}")
print(f"  Learning rate:       {dpo_config.learning_rate}")
print(f"  Beta (KL penalty):   {dpo_config.beta}")
print(f"  Max length:          {dpo_config.max_length}")
print(f"  Max prompt length:   {dpo_config.max_prompt_length}")

In [ ]:
# Initialize the DPO Trainer
# When using PEFT (LoRA), DPOTrainer automatically uses the base model weights
# as the reference model -- no need to load a separate copy!

dpo_trainer = DPOTrainer(
    model=model,                 # Policy model (will be trained)
    args=dpo_config,             # Training configuration
    train_dataset=train_dataset, # Training data (8 examples)
    eval_dataset=eval_dataset,   # Evaluation data (2 examples)
    tokenizer=tokenizer,         # Tokenizer for encoding text
    peft_config=peft_config,     # LoRA configuration
)

print("DPO Trainer initialized!")
print(f"\nTrainable parameters:")
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"  Trainable: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"  Total:     {total_params:,}")

In [ ]:
# Run DPO training!
# This should take approximately 5-10 minutes on a T4 GPU

import time

print("Starting DPO training...")
print("=" * 60)
start_time = time.time()

train_result = dpo_trainer.train()

elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"Training complete!")
print(f"  Time: {elapsed:.0f} seconds ({elapsed/60:.1f} minutes)")
print(f"  Final loss: {train_result.training_loss:.4f}")

In [ ]:
# Plot training loss to visualize how the model learned

import matplotlib.pyplot as plt

# Extract loss values from training log history
logs = dpo_trainer.state.log_history
train_losses = [(l["step"], l["loss"]) for l in logs if "loss" in l]

if train_losses:
    steps, losses = zip(*train_losses)
    
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, "b-o", markersize=4, label="DPO Loss")
    plt.xlabel("Training Step", fontsize=12)
    plt.ylabel("DPO Loss", fontsize=12)
    plt.title("DPO Training Loss Over Time", fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
    
    print(f"\nLoss progression: {losses[0]:.4f} -> {losses[-1]:.4f}")
    print(f"Total reduction: {losses[0] - losses[-1]:.4f}")
else:
    print("No training losses recorded. Check logging_steps configuration.")

In [ ]:
# Save the DPO-trained LoRA adapter

output_dir = "./alwajeez-dpo-adapter"
dpo_trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to: {output_dir}")

# List saved files
import os
saved_files = os.listdir(output_dir)
print(f"\nSaved files:")
for f in sorted(saved_files):
    size = os.path.getsize(os.path.join(output_dir, f))
    if size > 1024 * 1024:
        print(f"  {f} ({size / 1024 / 1024:.1f} MB)")
    else:
        print(f"  {f} ({size / 1024:.1f} KB)")

---

## Section 6: Compare DPO vs Base

### Comparison Methodology

Now we will compare the DPO-trained model against the base model to see if alignment
training made a meaningful difference. We will:

1. **Load both models**: The DPO-trained model (with LoRA adapter) and the base model
   (without adapter)
2. **Test on domain prompts**: Use prompts related to our system's domain
3. **Compare side-by-side**: Print responses from both models for visual inspection

We expect the DPO model to produce responses that are:
- More specific and detailed (like our "chosen" examples)
- Less vague (avoiding patterns from our "rejected" examples)
- Better structured with concrete details

Note: With only 10 preference pairs, improvements may be subtle. In production, you would
use hundreds or thousands of examples for stronger alignment.

In [ ]:
# ============================================================
# Section 6: Compare DPO vs Base
# ============================================================

from peft import PeftModel

# The DPO-trained model is already in memory as `model` with LoRA adapters
# For the base model comparison, we can disable the adapters temporarily

# First, get the model with the DPO adapter applied
dpo_model = dpo_trainer.model

print("Models ready for comparison:")
print("  - Base model: TinyLlama-1.1B-Chat (LoRA adapters disabled)")
print("  - DPO model:  TinyLlama-1.1B-Chat + DPO LoRA adapter")

In [ ]:
# Define test prompts about our domain topics
# These cover different aspects of the Alwajeez AI system

test_prompts = [
    "What is ZATCA and what are its e-invoicing requirements?",
    "How does the ReAct agent process a user request?",
    "What is the VAT rate in Saudi Arabia and what are the categories?",
    "How do I deploy the Agentic AI System?",
]

def format_prompt(question):
    """Format a question with the TinyLlama chat template."""
    return (
        f"<|system|>\n"
        f"You are Alwajeez AI, a helpful assistant for Alwajeez Tech's Agentic AI System. "
        f"You provide accurate, specific, and well-structured answers about ZATCA compliance, "
        f"VAT regulations, and the system's technical architecture.</s>\n"
        f"<|user|>\n"
        f"{question}</s>\n"
        f"<|assistant|>\n"
    )

print(f"Prepared {len(test_prompts)} test prompts:")
for i, p in enumerate(test_prompts):
    print(f"  {i+1}. {p}")

In [ ]:
# Generate responses from both models

def generate_from_model(model, tokenizer, prompt, max_new_tokens=250):
    """Generate a response using the given model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


print("Generating responses from both models...")
print("(This may take a minute)\n")

base_responses = []
dpo_responses = []

for i, question in enumerate(test_prompts):
    prompt = format_prompt(question)
    print(f"  Generating for prompt {i+1}/{len(test_prompts)}...")
    
    # Generate with base model (disable LoRA adapters)
    dpo_model.disable_adapter_layers()
    base_resp = generate_from_model(dpo_model, tokenizer, prompt)
    base_responses.append(base_resp)
    
    # Generate with DPO model (enable LoRA adapters)
    dpo_model.enable_adapter_layers()
    dpo_resp = generate_from_model(dpo_model, tokenizer, prompt)
    dpo_responses.append(dpo_resp)

print("\nDone! All responses generated.")

In [ ]:
# Print side-by-side comparison

for i, prompt in enumerate(test_prompts):
    print(f"{'=' * 70}")
    print(f"PROMPT {i+1}: {prompt}")
    print(f"{'=' * 70}")
    
    print(f"\n--- BASE MODEL ({len(base_responses[i].split())} words) ---")
    print(base_responses[i])
    
    print(f"\n--- DPO MODEL ({len(dpo_responses[i].split())} words) ---")
    print(dpo_responses[i])
    
    print()  # Blank line between examples

---

## Section 7: Alignment Evaluation

### Quantitative Evaluation Metrics

Visual comparison is useful but subjective. To properly evaluate alignment, we need
**quantitative metrics**. We will measure three aspects:

1. **Accuracy**: Does the response contain expected factual keywords? For example, a response
   about ZATCA should mention "Zakat", "Tax", "e-invoicing", "Phase 1", "Phase 2", etc.

2. **Specificity**: Does the response avoid vague, unhelpful language? Words like "stuff",
   "things", "basically", "kind of" indicate low-quality responses.

3. **Completeness**: Is the response sufficiently detailed? Very short responses (under 20
   words) are rarely complete. Well-structured responses with 50+ words tend to be more
   useful.

These are **heuristic metrics** -- they approximate quality but are not perfect. In
production, you would supplement these with human evaluation or LLM-as-judge approaches.

In [ ]:
# ============================================================
# Section 7: Alignment Evaluation
# ============================================================

# Define scoring functions for quantitative evaluation

def score_accuracy(response, keywords):
    """
    Score based on presence of expected keywords in the response.
    
    Args:
        response: The model's response text
        keywords: List of expected factual keywords
    
    Returns:
        Float between 0.0 and 1.0 (fraction of keywords found)
    """
    if not keywords:
        return 0.0
    found = sum(1 for kw in keywords if kw.lower() in response.lower())
    return found / len(keywords)


def score_specificity(response):
    """
    Score based on absence of vague, unhelpful language.
    
    Penalizes responses that use imprecise wording like "stuff", "things",
    "basically", etc. A perfectly specific response scores 1.0.
    
    Returns:
        Float between 0.0 and 1.0
    """
    vague_phrases = [
        "stuff", "things", "etc", "and so on", "basically",
        "kind of", "sort of", "whatever", "something like",
    ]
    vague_count = sum(1 for p in vague_phrases if p in response.lower())
    return max(0.0, 1.0 - vague_count * 0.2)


def score_completeness(response):
    """
    Score based on response length and structure.
    
    Longer, more detailed responses are considered more complete.
    This is a rough heuristic -- not all long responses are good.
    
    Returns:
        Float between 0.2 and 1.0
    """
    words = len(response.split())
    if words < 20:
        return 0.2
    elif words < 50:
        return 0.5
    elif words < 100:
        return 0.8
    else:
        return 1.0


# Verify scoring functions work correctly
print("Scoring function tests:")
print(f"  accuracy('ZATCA handles VAT at 15%', ['ZATCA', '15%', 'e-invoicing']) = "
      f"{score_accuracy('ZATCA handles VAT at 15%', ['ZATCA', '15%', 'e-invoicing']):.2f}")
print(f"  specificity('It handles stuff and things') = "
      f"{score_specificity('It handles stuff and things'):.2f}")
print(f"  specificity('ZATCA mandates e-invoicing in two phases') = "
      f"{score_specificity('ZATCA mandates e-invoicing in two phases'):.2f}")
print(f"  completeness('Short answer.') = "
      f"{score_completeness('Short answer.'):.2f}")
print(f"  completeness('{'word ' * 80}') = "
      f"{score_completeness('word ' * 80):.2f}")

In [ ]:
# Define evaluation set with expected keywords per question
# These keywords represent the factual content we expect in a good response

eval_set = [
    {
        "prompt": "What is ZATCA and what are its e-invoicing requirements?",
        "keywords": [
            "Zakat", "Tax", "Customs", "15%", "e-invoicing",
            "Phase 1", "Phase 2", "Fatoora", "XML", "QR",
        ],
    },
    {
        "prompt": "How does the ReAct agent process a user request?",
        "keywords": [
            "Thought", "Action", "Observation", "tool", "Ollama",
            "loop", "iteration", "FINAL ANSWER",
        ],
    },
    {
        "prompt": "What is the VAT rate in Saudi Arabia and what are the categories?",
        "keywords": [
            "15%", "5%", "2020", "zero-rated", "exempt",
            "standard", "375,000", "SAR",
        ],
    },
    {
        "prompt": "How do I deploy the Agentic AI System?",
        "keywords": [
            "Docker", "docker-compose", "8000", "11434", "5678",
            "Ollama", "FastAPI", "n8n", "health",
        ],
    },
]

print(f"Evaluation set: {len(eval_set)} questions")
for item in eval_set:
    print(f"  - {item['prompt']}")
    print(f"    Expected keywords: {', '.join(item['keywords'][:5])}...")

In [ ]:
# Score both models on all metrics and create a comparison table

results = []

for i, item in enumerate(eval_set):
    base_resp = base_responses[i]
    dpo_resp = dpo_responses[i]
    keywords = item["keywords"]
    
    results.append({
        "prompt": item["prompt"][:50] + "...",
        "base_accuracy": score_accuracy(base_resp, keywords),
        "dpo_accuracy": score_accuracy(dpo_resp, keywords),
        "base_specificity": score_specificity(base_resp),
        "dpo_specificity": score_specificity(dpo_resp),
        "base_completeness": score_completeness(base_resp),
        "dpo_completeness": score_completeness(dpo_resp),
    })

# Print formatted comparison table
print(f"{'=' * 100}")
print(f"ALIGNMENT EVALUATION: BASE MODEL vs DPO MODEL")
print(f"{'=' * 100}")
print()

# Header
print(f"{'Prompt':<55} {'Accuracy':^18} {'Specificity':^18} {'Completeness':^18}")
print(f"{'':<55} {'Base':>8} {'DPO':>8} {'Base':>8} {'DPO':>8} {'Base':>8} {'DPO':>8}")
print(f"{'-' * 55} {'-' * 18} {'-' * 18} {'-' * 18}")

for r in results:
    print(
        f"{r['prompt']:<55} "
        f"{r['base_accuracy']:>7.2f} {r['dpo_accuracy']:>8.2f} "
        f"{r['base_specificity']:>8.2f} {r['dpo_specificity']:>8.2f} "
        f"{r['base_completeness']:>8.2f} {r['dpo_completeness']:>8.2f}"
    )

print(f"{'-' * 55} {'-' * 18} {'-' * 18} {'-' * 18}")

In [ ]:
# Print overall averages and percentage improvement

def avg(values):
    return sum(values) / len(values) if values else 0

# Compute averages
avg_base_acc = avg([r["base_accuracy"] for r in results])
avg_dpo_acc = avg([r["dpo_accuracy"] for r in results])
avg_base_spec = avg([r["base_specificity"] for r in results])
avg_dpo_spec = avg([r["dpo_specificity"] for r in results])
avg_base_comp = avg([r["base_completeness"] for r in results])
avg_dpo_comp = avg([r["dpo_completeness"] for r in results])

# Compute improvements
def pct_improvement(base, dpo):
    if base == 0:
        return float('inf') if dpo > 0 else 0
    return ((dpo - base) / base) * 100

print(f"\n{'=' * 60}")
print(f"OVERALL AVERAGES")
print(f"{'=' * 60}")
print(f"")
print(f"{'Metric':<20} {'Base':>10} {'DPO':>10} {'Improvement':>15}")
print(f"{'-' * 60}")
print(f"{'Accuracy':<20} {avg_base_acc:>10.3f} {avg_dpo_acc:>10.3f} "
      f"{pct_improvement(avg_base_acc, avg_dpo_acc):>+14.1f}%")
print(f"{'Specificity':<20} {avg_base_spec:>10.3f} {avg_dpo_spec:>10.3f} "
      f"{pct_improvement(avg_base_spec, avg_dpo_spec):>+14.1f}%")
print(f"{'Completeness':<20} {avg_base_comp:>10.3f} {avg_dpo_comp:>10.3f} "
      f"{pct_improvement(avg_base_comp, avg_dpo_comp):>+14.1f}%")
print(f"{'-' * 60}")

# Overall composite score
base_overall = (avg_base_acc + avg_base_spec + avg_base_comp) / 3
dpo_overall = (avg_dpo_acc + avg_dpo_spec + avg_dpo_comp) / 3
print(f"{'OVERALL':<20} {base_overall:>10.3f} {dpo_overall:>10.3f} "
      f"{pct_improvement(base_overall, dpo_overall):>+14.1f}%")
print()

if dpo_overall > base_overall:
    print(f"The DPO model shows improvement over the base model.")
    print(f"DPO alignment training successfully shifted the model toward preferred responses.")
else:
    print(f"Note: With only 10 preference pairs and a small model, improvements may be subtle.")
    print(f"In production, use hundreds of preference pairs for stronger alignment.")

---

## Section 8: PPO Overview (Demonstrative)

### PPO: Proximal Policy Optimization

While DPO is our recommended approach for alignment, it is important to understand **PPO
(Proximal Policy Optimization)** -- the traditional RLHF method used by OpenAI for ChatGPT
and Anthropic for Claude (among other techniques).

### How PPO Works for RLHF

The PPO-based RLHF pipeline has four components:

1. **Policy Model**: The language model being trained (generates responses)
2. **Reference Model**: A frozen copy of the policy (prevents divergence)
3. **Reward Model**: A separate model trained on human preferences to predict which
   response a human would prefer. It outputs a scalar reward score.
4. **Value Head**: An extra linear layer added to the policy model that estimates the
   expected future reward (used for advantage estimation in PPO)

### PPO Training Loop

```
For each batch:
  1. Generate responses from the policy model
  2. Score each response with the reward model -> reward_score
  3. Compute KL penalty: kl = log(policy_prob / reference_prob)
  4. Adjusted reward = reward_score - beta * kl
  5. Compute advantages using GAE (Generalized Advantage Estimation)
  6. Update policy using PPO clipped objective
  7. Update value head to better predict rewards
```

### Why PPO is More Complex

- Requires training a **separate reward model** on preference data first
- Uses an **online RL loop** (generate -> score -> update) which is slower than supervised training
- Can suffer from **reward hacking**: the policy finds responses that score high on the reward
  model but aren't actually good (exploiting reward model weaknesses)
- Requires careful hyperparameter tuning (clipping, KL coefficient, value function coefficient)
- Needs more GPU memory (4 models in memory: policy, reference, reward, value head)

### When to Use Each

| Scenario | Recommended |
|----------|-------------|
| Small dataset of static preferences | **DPO** |
| Need online learning from user feedback | **PPO** |
| Limited compute / T4 GPU | **DPO** |
| Large-scale production alignment | **PPO** (or RLHF variants) |
| First alignment experiment | **DPO** (simpler to set up) |
| Complex, multi-dimensional reward | **PPO** (custom reward function) |

In [ ]:
# ============================================================
# Section 8: PPO Overview (Demonstrative)
# ============================================================

# This code shows how PPO would be set up using TRL.
# NOTE: We do NOT run full PPO training here because:
#   1. It requires significantly more VRAM than a T4 provides
#   2. It requires a trained reward model
#   3. The training loop is much longer
# This is for educational purposes only.

print("PPO Setup (Demonstrative - NOT executed)")
print("=" * 50)
print()
print("The following shows how PPO training would be configured:")
print()

ppo_setup_code = """
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead

# Step 1: Configure PPO
ppo_config = PPOConfig(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    learning_rate=1e-5,        # Lower LR for RL stability
    batch_size=4,              # Batch of prompts per PPO step
    mini_batch_size=2,         # Mini-batch for PPO updates
    ppo_epochs=4,              # PPO update epochs per batch
    kl_penalty="kl",           # KL divergence penalty type
    init_kl_coef=0.2,          # Initial KL penalty coefficient
)

# Step 2: Load model with value head
# The value head is an extra linear layer that predicts expected reward
model = AutoModelForCausalLMWithValueHead.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

# Step 3: Set up reward model (or reward function)
# Option A: Trained reward model
#   reward_model = pipeline("text-classification", model="my-reward-model")
# Option B: Rule-based reward function
#   reward_fn = lambda response: score_based_on_rules(response)

# Step 4: PPO Training Loop
for epoch in range(num_epochs):
    for batch in dataloader:
        # 1. Generate responses from policy
        responses = model.generate(batch["input_ids"])
        
        # 2. Score with reward model
        rewards = [reward_model(resp) for resp in responses]
        
        # 3. Run PPO update step
        stats = ppo_trainer.step(
            queries=batch["input_ids"],
            responses=responses,
            rewards=rewards,
        )
        # PPO internally:
        #   - Computes log probs from policy and reference
        #   - Computes KL penalty
        #   - Computes advantages using GAE
        #   - Updates policy with clipped PPO objective
        #   - Updates value head
"""

print(ppo_setup_code)

In [ ]:
# PPO vs DPO Comparison Table

print("\nPPO vs DPO Comparison:")
print(f"{'=' * 85}")
print(f"{'Aspect':<25} {'DPO':<30} {'PPO':<30}")
print(f"{'-' * 85}")
print(f"{'Reward Model':<25} {'Not needed':<30} {'Required (separate model)':<30}")
print(f"{'Compute Cost':<25} {'~2x SFT':<30} {'~4-10x SFT':<30}")
print(f"{'Training Stability':<25} {'Very stable':<30} {'Can be unstable':<30}")
print(f"{'Flexibility':<25} {'Fixed offline preferences':<30} {'Dynamic online rewards':<30}")
print(f"{'Implementation':<25} {'Simple (supervised-like)':<30} {'Complex (RL loop)':<30}")
print(f"{'Memory (T4 GPU)':<25} {'Fits with QLoRA':<30} {'Tight / may not fit':<30}")
print(f"{'Reward Hacking Risk':<25} {'Low':<30} {'Moderate to High':<30}")
print(f"{'Data Requirement':<25} {'Preference pairs':<30} {'Prompts + reward model':<30}")
print(f"{'Online Learning':<25} {'No (offline only)':<30} {'Yes (can generate new data)':<30}")
print(f"{'Used By':<25} {'Llama 2/3, Zephyr':<30} {'ChatGPT (early), InstructGPT':<30}")
print(f"{'=' * 85}")
print()
print("Recommendation for Alwajeez Tech:")
print("  Use DPO for initial alignment experiments. It is simpler, more stable, and fits")
print("  within T4 GPU constraints. Consider PPO only if you need online learning from")
print("  real-time user feedback at scale.")

---

## Section 9: Simple Reward Function

### Building Rule-Based Reward Functions

A **reward function** assigns a numerical score to a model response, measuring how "good"
it is. Reward functions are used in two ways:

1. **Evaluation**: Score model outputs to measure alignment quality (what we do here)
2. **PPO Training**: Provide rewards for the RL training loop (if you use PPO)

There are three types of reward functions:

| Type | Description | Example |
|------|-------------|--------|
| **Rule-based** | Hand-crafted rules checking for specific patterns | Check for keywords, penalize vagueness |
| **Learned (classifier)** | A model trained on preference data to predict quality | Bradley-Terry reward model |
| **LLM-as-Judge** | Use a large LLM to score responses | GPT-4 scoring on a 1-5 scale |

For this lab, we build a **rule-based reward function** that scores responses on:
- Specificity (absence of vague language)
- Structure (presence of numbered lists / bullet points)
- Factual content (domain-specific keywords and data points)
- Adequate length (sufficient detail)

Rule-based rewards are transparent, fast, and don't require additional models.
They are a good starting point and can be combined with learned rewards later.

In [ ]:
# ============================================================
# Section 9: Simple Reward Function
# ============================================================

def reward_function(response, prompt):
    """
    Rule-based reward function scoring response quality.
    
    This function evaluates four aspects of a response:
    1. Specificity: Penalizes vague, imprecise language
    2. Structure: Rewards organized responses with lists/numbering
    3. Factual content: Rewards presence of domain-specific terms
    4. Length: Rewards adequate response length
    
    Args:
        response: The model's response text
        prompt: The input prompt (for context-aware scoring)
    
    Returns:
        Float between 0.0 and 1.0 (normalized reward score)
    """
    score = 0.0
    
    # --- Component 1: Specificity (max 1.0) ---
    # Penalize vague, imprecise language that indicates low-quality responses
    vague = ["stuff", "things", "etc", "basically", "kind of",
             "sort of", "whatever", "and so on"]
    vague_count = sum(1 for v in vague if v in response.lower())
    specificity_score = max(0, 1.0 - vague_count * 0.3)
    score += specificity_score
    
    # --- Component 2: Structure (max 0.5) ---
    # Reward numbered lists and structured formatting
    has_numbering = any(
        f"{i})" in response or f"{i}." in response or f"{i}) " in response
        for i in range(1, 6)
    )
    if has_numbering:
        score += 0.5
    
    # --- Component 3: Factual Content (max 1.0) ---
    # Reward specific technical terms and data points from our domain
    factual_markers = [
        "port", "15%", "ZATCA", "Ollama", "FastAPI",
        "ReAct", "8000", "11434", "5678", "Docker",
        "Phase 1", "Phase 2", "Fatoora", "QLoRA", "LoRA",
    ]
    factual_count = sum(1 for m in factual_markers if m in response)
    factual_score = min(1.0, factual_count * 0.25)
    score += factual_score
    
    # --- Component 4: Length (max 1.0) ---
    # Reward adequate response length (more detail is generally better)
    words = len(response.split())
    if words > 30:
        score += 0.5
    if words > 80:
        score += 0.5
    
    # Normalize to 0-1 range (max possible = 1.0 + 0.5 + 1.0 + 1.0 = 3.5)
    normalized_score = score / 3.5
    
    return normalized_score


print("Reward function defined with 4 components:")
print("  1. Specificity (max 1.0) - penalizes vague language")
print("  2. Structure   (max 0.5) - rewards numbered lists")
print("  3. Factual     (max 1.0) - rewards domain-specific terms")
print("  4. Length      (max 1.0) - rewards adequate detail")
print("  Total max raw score: 3.5, normalized to 0.0 - 1.0")

In [ ]:
# Test the reward function on sample good vs bad responses

good_response = (
    "ZATCA stands for Zakat, Tax and Customs Authority. It regulates VAT at 15% in "
    "Saudi Arabia and mandates e-invoicing through Phase 1 (generation) and Phase 2 "
    "(integration with Fatoora platform). Businesses must: 1) Generate XML invoices with "
    "QR codes. 2) Integrate with ZATCA's Fatoora portal for real-time clearance. "
    "3) Register for VAT if annual revenue exceeds SAR 375,000."
)

bad_response = "ZATCA is a government thing that handles taxes and stuff."

medium_response = (
    "ZATCA is the Zakat, Tax and Customs Authority in Saudi Arabia. They handle VAT "
    "collection at 15% and manage e-invoicing requirements for businesses."
)

print("Reward Function Test Results:")
print("=" * 60)

print(f"\nGood response ({len(good_response.split())} words):")
print(f"  '{good_response[:80]}...'")
print(f"  Score: {reward_function(good_response, 'What is ZATCA?'):.3f}")

print(f"\nMedium response ({len(medium_response.split())} words):")
print(f"  '{medium_response[:80]}...'")
print(f"  Score: {reward_function(medium_response, 'What is ZATCA?'):.3f}")

print(f"\nBad response ({len(bad_response.split())} words):")
print(f"  '{bad_response}'")
print(f"  Score: {reward_function(bad_response, 'What is ZATCA?'):.3f}")

print(f"\nThe reward function correctly ranks: Good > Medium > Bad")
print(f"This validates it can distinguish response quality levels.")

In [ ]:
# Score all DPO model outputs vs base model outputs using the reward function

print("Reward Function Scoring: Base vs DPO Model")
print("=" * 70)
print()

base_rewards = []
dpo_rewards = []

for i, question in enumerate(test_prompts):
    base_reward = reward_function(base_responses[i], question)
    dpo_reward = reward_function(dpo_responses[i], question)
    base_rewards.append(base_reward)
    dpo_rewards.append(dpo_reward)
    
    winner = "DPO" if dpo_reward > base_reward else "Base" if base_reward > dpo_reward else "Tie"
    
    print(f"Q{i+1}: {question[:50]}...")
    print(f"    Base reward: {base_reward:.3f}  |  DPO reward: {dpo_reward:.3f}  |  Winner: {winner}")
    print()

# Overall statistics
avg_base_reward = sum(base_rewards) / len(base_rewards)
avg_dpo_reward = sum(dpo_rewards) / len(dpo_rewards)
dpo_wins = sum(1 for b, d in zip(base_rewards, dpo_rewards) if d > b)
base_wins = sum(1 for b, d in zip(base_rewards, dpo_rewards) if b > d)
ties = sum(1 for b, d in zip(base_rewards, dpo_rewards) if b == d)

print(f"{'=' * 70}")
print(f"OVERALL REWARD SCORES")
print(f"{'=' * 70}")
print(f"  Average Base reward:  {avg_base_reward:.3f}")
print(f"  Average DPO reward:   {avg_dpo_reward:.3f}")
if avg_base_reward > 0:
    print(f"  Improvement:          {((avg_dpo_reward - avg_base_reward) / avg_base_reward) * 100:+.1f}%")
print(f"")
print(f"  Win/Loss/Tie: DPO wins {dpo_wins}, Base wins {base_wins}, Ties {ties}")
print(f"  DPO win rate: {dpo_wins / len(test_prompts) * 100:.0f}%")

---

## Summary: What We Learned

### Key Takeaways

1. **RLHF** is a paradigm for aligning language models with human preferences. The full
   pipeline (SFT -> Reward Model -> PPO) is powerful but complex.

2. **DPO** provides a simpler alternative that directly optimizes on preference pairs
   without needing a separate reward model. It is:
   - Mathematically equivalent to PPO under certain assumptions
   - Much simpler to implement (supervised learning, not RL)
   - More stable during training
   - Feasible on a single T4 GPU with QLoRA

3. **Preference data quality** matters more than quantity. Our 10 carefully crafted
   preference pairs demonstrate the concept, but production systems typically use
   hundreds or thousands of examples.

4. **The beta parameter** controls the trade-off between alignment strength and
   staying close to the original model. Lower beta = more alignment, higher beta =
   more conservative.

5. **Reward functions** can be rule-based (transparent, fast) or learned (more flexible).
   They are essential for evaluation even if you use DPO for training.

### Comparison of Alignment Techniques

| Technique | Complexity | Data Required | Compute | Best For |
|-----------|-----------|--------------|---------|----------|
| SFT | Low | Instruction-output pairs | 1x | Teaching new knowledge |
| DPO | Medium | Preference triples | ~2x | Alignment with offline preferences |
| PPO (RLHF) | High | Prompts + reward model | ~4-10x | Online alignment at scale |
| ORPO | Medium | Preference triples | ~1.5x | Combined SFT + alignment |
| KTO | Medium | Good/bad labels (no pairs) | ~2x | When pairing is difficult |

### Next Steps for Alwajeez Tech

1. **Scale preference data**: Collect 100+ preference pairs from domain experts
2. **Combine SFT + DPO**: First fine-tune on domain data (Module 3), then align with DPO
3. **Deploy aligned model**: Merge the DPO adapter, convert to GGUF, and import into Ollama
4. **Continuous improvement**: Collect user feedback to create new preference pairs
5. **Evaluate with LLM-as-Judge**: Use a stronger model to evaluate response quality

### Connection to Our Agentic AI System

The alignment pipeline fits into the Alwajeez Tech system as follows:

```
Base Model (llama3.2:3b)
    |-- QLoRA Fine-tuning (Module 3) -> Domain knowledge
    |-- DPO Alignment (Module 4) -> Response quality
    |-> Merge adapters -> Convert to GGUF -> Import to Ollama
    |-> FastAPI serves the aligned model via /v1/chat/completions
    |-> ReAct agent uses the aligned model for reasoning
```

The aligned model will produce more accurate, specific, and well-structured responses
throughout the entire system -- from direct chat to agent reasoning steps.